# Basic Tools

In [10]:
from dotenv import load_dotenv
load_dotenv()

from pprint import pprint

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage

In [ ]:
# Create the tools

@tool
def discount_amount(price: float, discount_percent: float) -> float:
    """Calculate discount amount from price and discount percentage."""
    return price * (discount_percent / 100)

@tool("tax_amount")
def tool1(subtotal: float, tax_percent: float) -> float:
    """Calculate the amount from subtotal and tax percentage."""
    return subtotal * (tax_percent / 100)

@tool("final_price", description="Calculate final price after discount and adding tax")
def tool2(price: float, discount_percent: float, tax_percent: float) -> float:
    discount = price * (discount_percent / 100)
    subtotal = price - discount
    tax = subtotal * (tax_percent / 100)
    return subtotal + tax

In [6]:
# Test tools

discount_amount.invoke({"price":120, "discount_percent":10})

tool1.invoke({"subtotal":180, "tax_percent":12})

tool2.invoke({"price": 120, "discount_percent": 10, "tax_percent": 12})

120.96

In [8]:
# Add tools to an agent

MODEL_NAME = "openai:gpt-5-nano"
SYSTEM_PROMPT = """You are a helpful shopping assistant.
Use your tools to calculate discounts, taxes, and final prices accurately.
"""

agent = create_agent(
    model=MODEL_NAME,
    tools=[discount_amount, tool1, tool2],
    system_prompt=SYSTEM_PROMPT
)

In [11]:
# Ask the agent a question

question = HumanMessage(
    content="If a product costs 250, has a 15% discount, and then 12% tax, what is the final price?"
)

response = agent.invoke(
    input={"messages": [question]}
)

pprint(response["messages"][-1].content)

('Final price: 238.00\n'
 '\n'
 'Breakdown:\n'
 '- Original price: 250.00\n'
 '- Discount (15%): -37.50\n'
 '- Price after discount: 212.50\n'
 '- Tax (12%) on discounted price: +25.50\n'
 '- Final price: 238.00')


In [ ]:
# Streaming

for event in agent.stream(
    {"messages":[question]},
    stream_mode="updates"
):
    pprint(event)
    print("-" * 80)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 610, 'prompt_tokens': 244, 'total_tokens': 854, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DEeQ9ya3IzSKoHSz4ajXcmyM2Rnfb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019caa56-cd8a-7c31-b721-d286f036ad34-0', tool_calls=[{'name': 'final_price', 'args': {'price': 250, 'discount_percent': 15, 'tax_percent': 12}, 'id': 'call_OqgE5tVBvysZnKB9U4N6vF3t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 244, 'output_tokens': 610, 'total_tokens': 854, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_to